In [21]:
import csv
import json
import random
import numpy as np

random.seed(42)

with open("queries.txt", "r") as f:
    query_to_id = {}
    id_to_query = {}

    for line in f:
        query_id, query = line.strip().split("\t", 1)
        query_id = int(query_id[1:])
        query_to_id[query] = query_id
        id_to_query[query_id] = query

with open("dataset.json", "r") as f:
    dataset = json.load(f)

documents = {}
num_results = {}

for query, results in dataset.items():
    qid = query_to_id[query]
    documents[qid] = []

    for i, result in enumerate(results, start=1):
        result["doc_id"] = f"D{i}"
        documents[qid].append(result)

    num_results[qid] = len(results)

relevance = {}

with open("ground_truth.txt", "r", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        qid = int(row["query_id"][1:])
        doc_id = row["doc_id"]
        rank = int(row["relevance_rank"])

        relevance[(qid, doc_id)] = rank

def simulate_clicks(query_id, results, max_clicks=3, decay=0.3):
    n = len(results)
    weights = np.exp(-decay * np.arange(n))
    probabilities = weights / weights.sum()

    num_clicks = random.randint(1, min(max_clicks, n))
    indices = np.random.choice(n, size=num_clicks, replace=False, p=probabilities)

    return [results[i]["doc_id"] for i in indices]

def generate_preferences(num_sessions=100, max_clicks=3, decay=0.3):
    preferences = []

    for query_id, results in documents.items():
        for session_id in range(num_sessions):
            clicked_ids = set(simulate_clicks(query_id, results, max_clicks=max_clicks, decay=decay))

            for i, result in enumerate(results):
                clicked_id = result["doc_id"]
                if clicked_id not in clicked_ids:
                    continue

                for j in range(i):
                    above_id = results[j]["doc_id"]

                    if above_id not in clicked_ids:
                        preferences.append({
                            "query_id": query_id,
                            "session_id": session_id,
                            "preferred_doc": clicked_id,
                            "less_preferred_doc": above_id
                        })

    return preferences

preferences = generate_preferences(num_sessions=100)

with open("preferences.csv", "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "query_id",
            "session_id",
            "preferred_doc",
            "less_preferred_doc"
        ]
    )
    writer.writeheader()
    writer.writerows(preferences)

print(f"Generated {len(preferences)} pairwise preferences.")
print(preferences[:10])

# query = "How does gradient descent work in machine learning?"

# query_id = query_to_id[query]
# results = documents[query_id]

# clicked_docs = simulate_clicks(query_id, results)

# print("Clicked documents:", clicked_docs)

# for doc_id in clicked_docs:
#     result = next(r for r in results if r["doc_id"] == doc_id)
#     print(doc_id, result["title"])


Generated 8393 pairwise preferences.
[{'query_id': 1, 'session_id': 0, 'preferred_doc': 'D6', 'less_preferred_doc': 'D2'}, {'query_id': 1, 'session_id': 0, 'preferred_doc': 'D6', 'less_preferred_doc': 'D3'}, {'query_id': 1, 'session_id': 0, 'preferred_doc': 'D6', 'less_preferred_doc': 'D4'}, {'query_id': 1, 'session_id': 0, 'preferred_doc': 'D6', 'less_preferred_doc': 'D5'}, {'query_id': 1, 'session_id': 0, 'preferred_doc': 'D8', 'less_preferred_doc': 'D2'}, {'query_id': 1, 'session_id': 0, 'preferred_doc': 'D8', 'less_preferred_doc': 'D3'}, {'query_id': 1, 'session_id': 0, 'preferred_doc': 'D8', 'less_preferred_doc': 'D4'}, {'query_id': 1, 'session_id': 0, 'preferred_doc': 'D8', 'less_preferred_doc': 'D5'}, {'query_id': 1, 'session_id': 0, 'preferred_doc': 'D8', 'less_preferred_doc': 'D7'}, {'query_id': 1, 'session_id': 1, 'preferred_doc': 'D3', 'less_preferred_doc': 'D1'}]
